In [ ]:
!pip install timm
!pip install transformers

In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import transforms
import timm
from PIL import Image
from transformers import VivitForVideoClassification, VivitConfig
import random

# Directories for real and manipulated videos
real_videos_dir = "/kaggle/input/deep-fake-detection-dfd-entire-original-dataset/DFD_original sequences"
manipulated_videos_dir = "/kaggle/input/deep-fake-detection-dfd-entire-original-dataset/DFD_manipulated_sequences/DFD_manipulated_sequences"

# Output directories for extracted video clips
output_real_dir = "/kaggle/working/clips/real"
output_manipulated_dir = "/kaggle/working/clips/manipulated"

# Ensure output directories exist
os.makedirs(output_real_dir, exist_ok=True)
os.makedirs(output_manipulated_dir, exist_ok=True)

def extract_video_clips(videos_dir, output_dir, label, max_videos=50, frames_per_clip=16):
    """Extract video clips with multiple frames for temporal analysis"""
    video_files = [f for f in os.listdir(videos_dir) if f.endswith(('.mp4', '.avi', '.mov', '.mkv'))]
    video_files = video_files[:max_videos]  # Limit to max_videos

    for video_file in video_files:
        video_path = os.path.join(videos_dir, video_file)
        cap = cv2.VideoCapture(video_path)
        
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        fps = int(cap.get(cv2.CAP_PROP_FPS))
        
        # Extract multiple clips from each video
        clips_per_video = 3  # Extract 3 clips per video
        for clip_idx in range(clips_per_video):
            # Random start frame for each clip
            start_frame = random.randint(0, max(0, total_frames - frames_per_clip * 2))
            cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
            
            frames = []
            for frame_idx in range(frames_per_clip):
                ret, frame = cap.read()
                if not ret:
                    break
                frames.append(frame)
            
            # Save clip as numpy array if we have enough frames
            if len(frames) == frames_per_clip:
                clip_filename = f"{label}_{video_file}_clip{clip_idx}.npy"
                clip_path = os.path.join(output_dir, clip_filename)
                frames_array = np.array(frames)
                np.save(clip_path, frames_array)
        
        cap.release()

# Extract video clips from 100 real and 100 manipulated videos
extract_video_clips(real_videos_dir, output_real_dir, "real", max_videos=100)
extract_video_clips(manipulated_videos_dir, output_manipulated_dir, "manipulated", max_videos=100)
print("Video clip extraction completed.")

# Custom Dataset for video clips
class VideoClipDataset(Dataset):
    def __init__(self, root_dir, transform=None, frames_per_clip=16):
        self.root_dir = root_dir
        self.transform = transform
        self.frames_per_clip = frames_per_clip
        
        # Get all clip files and labels
        self.clips = []
        self.labels = []
        
        for class_name in os.listdir(root_dir):
            class_dir = os.path.join(root_dir, class_name)
            if os.path.isdir(class_dir):
                label = 0 if class_name == 'real' else 1
                for clip_file in os.listdir(class_dir):
                    if clip_file.endswith('.npy'):
                        self.clips.append(os.path.join(class_dir, clip_file))
                        self.labels.append(label)
    
    def __len__(self):
        return len(self.clips)
    
    def __getitem__(self, idx):
        # Load video clip
        clip_path = self.clips[idx]
        frames = np.load(clip_path)  # Shape: (frames, height, width, channels)
        label = self.labels[idx]
        
        # Convert frames to tensor and apply transforms
        processed_frames = []
        for frame in frames:
            # Convert BGR to RGB
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame_pil = Image.fromarray(frame_rgb)
            
            if self.transform:
                frame_tensor = self.transform(frame_pil)
            else:
                frame_tensor = transforms.ToTensor()(frame_pil)
            
            processed_frames.append(frame_tensor)
        
        # Stack frames: (frames, channels, height, width)
        video_tensor = torch.stack(processed_frames)
        
        return video_tensor, label

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Define image transformations
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load the dataset
dataset_dir = "/kaggle/working/clips"  # Directory where clips are saved
dataset = VideoClipDataset(root=dataset_dir, transform=transform)

# Split dataset into training and validation sets
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=2)

# Create ViViT model
class SimpleViViT(nn.Module):
    def __init__(self, num_classes=2, num_frames=16):
        super(SimpleViViT, self).__init__()
        self.num_frames = num_frames
        
        # Use ViT as backbone for spatial features
        self.spatial_encoder = timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=0)
        
        # Temporal transformer layers
        self.temporal_pos_embedding = nn.Parameter(torch.randn(1, num_frames, 768))
        self.temporal_transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=768, nhead=12, batch_first=True),
            num_layers=4
        )
        
        # Classification head
        self.classifier = nn.Sequential(
            nn.LayerNorm(768),
            nn.Dropout(0.1),
            nn.Linear(768, num_classes)
        )
    
    def forward(self, x):
        # x shape: (batch, frames, channels, height, width)
        batch_size, num_frames, c, h, w = x.shape
        
        # Reshape for spatial processing: (batch*frames, channels, height, width)
        x = x.view(batch_size * num_frames, c, h, w)
        
        # Extract spatial features using ViT
        spatial_features = self.spatial_encoder(x)  # (batch*frames, 768)
        
        # Reshape back: (batch, frames, 768)
        spatial_features = spatial_features.view(batch_size, num_frames, -1)
        
        # Add temporal positional encoding
        temporal_features = spatial_features + self.temporal_pos_embedding
        
        # Apply temporal transformer
        temporal_output = self.temporal_transformer(temporal_features)
        
        # Global average pooling over temporal dimension
        pooled_features = temporal_output.mean(dim=1)  # (batch, 768)
        
        # Classification
        output = self.classifier(pooled_features)
        
        return output

# Initialize model
model = SimpleViViT(num_classes=2, num_frames=16)
model.to(device)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.7)

# Training loop with early stopping
num_epochs = 20
best_val_accuracy = 0
patience = 5
patience_counter = 0

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    
    for batch_idx, (videos, labels) in enumerate(train_loader):
        videos, labels = videos.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(videos)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()
        
        if batch_idx % 10 == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}], Batch [{batch_idx}/{len(train_loader)}], Loss: {loss.item():.4f}')

    train_accuracy = 100 * correct_train / total_train
    print(f"Epoch [{epoch+1}/{num_epochs}], Avg Loss: {running_loss/len(train_loader):.4f}, Training Accuracy: {train_accuracy:.2f}%")

    # Validation
    model.eval()
    correct_val = 0
    total_val = 0
    with torch.no_grad():
        for videos, labels in val_loader:
            videos, labels = videos.to(device), labels.to(device)
            outputs = model(videos)
            _, predicted = torch.max(outputs, 1)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()

    val_accuracy = 100 * correct_val / total_val
    print(f"Validation Accuracy: {val_accuracy:.2f}%")

    # Save best model with early stopping
    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        torch.save(model.state_dict(), 'best_vivit_model.pth')
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print("Early stopping due to no improvement.")
            break
    scheduler.step()

print(f"Best Validation Accuracy: {best_val_accuracy:.2f}%")

# Evaluation
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Load the best model
model.load_state_dict(torch.load('best_vivit_model.pth', weights_only=True))
model.eval()

all_labels = []
all_predictions = []

with torch.no_grad():
    for videos, labels in val_loader:
        videos, labels = videos.to(device), labels.to(device)
        outputs = model(videos)
        _, predicted = torch.max(outputs, 1)

        all_labels.extend(labels.cpu().numpy())
        all_predictions.extend(predicted.cpu().numpy())

# Calculate classification metrics
print("Classification Report:")
print(classification_report(all_labels, all_predictions, target_names=['Real', 'Manipulated']))

# Accuracy
accuracy = accuracy_score(all_labels, all_predictions)
print(f"Accuracy: {accuracy * 100:.2f}%")

# Confusion matrix
cm = confusion_matrix(all_labels, all_predictions)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Real', 'Manipulated'], yticklabels=['Real', 'Manipulated'])
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.title('ViViT Confusion Matrix')
plt.show()

# Video prediction function
def predict_video_vivit(video_path, model, transform, device, frames_per_clip=16, num_clips=5):
    """Predict if a video is real or manipulated using ViViT"""
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    predictions = []
    
    for clip_idx in range(num_clips):
        # Random start frame for each clip
        start_frame = random.randint(0, max(0, total_frames - frames_per_clip * 2))
        cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
        
        frames = []
        for _ in range(frames_per_clip):
            ret, frame = cap.read()
            if not ret:
                break
            
            # Convert and transform frame
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame_pil = Image.fromarray(frame_rgb)
            frame_tensor = transform(frame_pil)
            frames.append(frame_tensor)
        
        if len(frames) == frames_per_clip:
            # Stack frames and add batch dimension
            video_tensor = torch.stack(frames).unsqueeze(0).to(device)
            
            # Make prediction
            with torch.no_grad():
                outputs = model(video_tensor)
                _, predicted = torch.max(outputs, 1)
                predictions.append(predicted.item())
    
    cap.release()
    
    # Majority vote
    real_count = sum(1 for p in predictions if p == 0)
    manipulated_count = sum(1 for p in predictions if p == 1)
    
    if real_count > manipulated_count:
        result = "Real"
        confidence = real_count / len(predictions)
    else:
        result = "Manipulated"
        confidence = manipulated_count / len(predictions)
    
    print(f"Result: {result} video (confidence: {confidence:.2f})")
    print(f"Clip predictions: {real_count} real, {manipulated_count} manipulated")
    
    return result

# Test transform (without augmentations)
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Test the model on sample videos
print("Testing Real Video:")
video_path = "/kaggle/input/deep-fake-detection-dfd-entire-original-dataset/DFD_original sequences/02__kitchen_still.mp4"
result = predict_video_vivit(video_path, model, test_transform, device)

print("\nTesting Manipulated Video:")
video_path = "/kaggle/input/deep-fake-detection-dfd-entire-original-dataset/DFD_manipulated_sequences/DFD_manipulated_sequences/01_20__walking_and_outside_surprised__OTGHOG4Z.mp4"
result = predict_video_vivit(video_path, model, test_transform, device)